# AI Security Framework Crosswalk: From Baseline Failure to Ordinal Ensemble

**Author:** Rock Lambros, University of Denver, COMP 4433

**Abstract.** This notebook traces the development of a 4-class ordinal classifier for AI security framework crosswalks, from a baseline that scored 0% on its most important class to a 3-model ensemble that broke through.

The crosswalk dataset contains 983 security controls from nine AI security frameworks connected by 5,813 edges. Expert annotators labeled 179 pairs on a 4-tier ordinal scale: Unrelated, Partial, Related, Equivalent. The v7c baseline reached 81.0% exact accuracy and 0.512 macro F1, but scored 0.000 F1 on the Equivalent class. The classifier never predicted that two controls from different frameworks meant the same thing.

The Open Common Requirements Enumeration (OpenCRE) database provided 13,519 pairs with expert-curated relationships and a hop-distance structure that maps naturally onto ordinal tiers. After removing 34 pairs overlapping the frozen test set, 6,200 clean pairs remained. Disagreement mining---scoring these through v7c and selecting the 673 where it conflicted with OpenCRE labels---produced the v8 training augmentation.

v8b expanded augmentation further, but DeBERTa-large collapsed to single-class prediction and the LightGBM stacker overfit to training accuracy of 1.000 against validation accuracy of 0.528. Both failures pointed to the same problem: noisy proxy labels amplified by a learnable second stage.

v_final stripped the pipeline down. Mapping-level deduplication removed 56% of text-pair contamination from validation. Three ordinal loss functions (KL-divergence with ordinal smoothing, CORN ordinal regression, focal loss) replaced standard cross-entropy. Softmax averaging across RoBERTa-large, DeBERTa-v3-base, and BGE-large-v1.5 replaced the stacker. The result: macro F1 rose to 0.558 (+4.6 pp), Equivalent F1 reached 0.400 (from 0.000), conformal coverage exceeded 90% on all four classes, and the ensemble scored all 4,001 edges in the crosswalk graph. The trained model is available on [HuggingFace](https://huggingface.co/rockCO78/ai-security-crosswalk-vfinal).

> **Plain English:** I built a tool that compares security controls across nine AI security standards and decides whether two controls from different frameworks are unrelated, partially related, related, or equivalent. The first version worked well overall (81% accuracy) but could not identify equivalent controls at all---it scored 0% on the class that matters most for compliance mapping.
>
> After discovering OpenCRE, a public database that already links these standards through shared requirements, I tried two ways to add its data to training. The first (disagreement mining) was promising but limited. The second (direct augmentation) caused the large model to collapse to a single prediction and the combiner to memorize the training data perfectly while failing on new examples. Both failures pointed in the same direction: strip the architecture down and remove the learnable combiner.
>
> The final version averages three models trained with loss functions that care about the ordering of the tiers. It gets Equivalent right for the first time. The trained model is on HuggingFace for anyone to use.

## 2 · Setup and Data Loading

All artifacts referenced below live in `data/processed/`, `results/sacred/`, and `runs/v7c_sacred/`. The training pipeline writes them as part of its sacred evaluation pass; this notebook only reads them. The separation ensures that re-running every cell in this notebook cannot change the numbers the classifier reports.

In [ ]:
# COMP 4433 approved libraries only. I import numpy, pandas, matplotlib,
# and seaborn for the bulk of the notebook; sklearn and statsmodels are
# imported lazily inside the cells that use them (section 6 ablation
# baselines and the conclusion ordinal demonstrator) so the grader can
# see exactly which cells exercise each library. Loading the core
# four up front fails fast if any are missing.
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
import seaborn as sns

# Silence a cosmetic matplotlib warning that fires when tight_layout is
# combined with annotated gridspec panels. The figures render correctly;
# the warning is advisory and pollutes notebook output otherwise.
warnings.filterwarnings(
    "ignore",
    message="This figure includes Axes that are not compatible with tight_layout",
)

# Resolve the repo root in a way that works whether the notebook is launched
# from the repo root, from notebooks/, or from a fresh unzipped submission
# folder. I walk upward looking for the data/processed directory rather than
# relying on a hard-coded relative path.
HERE = Path.cwd()
candidate = HERE
for _ in range(4):
    if (candidate / "data" / "processed").exists():
        break
    candidate = candidate.parent
REPO = candidate
REPO_ROOT = REPO.parent if REPO.name == "project1" else REPO
DATA = REPO / "data" / "processed"
SACRED = REPO / "results" / "sacred"
V7C = REPO / "runs" / "v7c_sacred"
assert DATA.exists(), f"could not locate data/processed starting from {HERE}"

# Seaborn theme. The paper context and bold titles match a scientific
# document rather than a slide deck. I set savefig DPI high enough that PDF
# and PNG exports remain crisp if the grader rerenders the notebook.
sns.set_theme(style="whitegrid", context="paper", font_scale=1.15)
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "axes.titleweight": "bold",
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "legend.frameon": False,
})

def jload(name):
    return json.loads((DATA / name).read_text())

def cload(name):
    return pd.read_csv(DATA / name)

# Graph artifacts. nodes.json and edges.json are the canonical long-form tables
# that describe the crosswalk. graph_stats.json is a small summary precomputed
# once at ingest time so the notebook does not have to reaggregate from scratch.
nodes = jload("nodes.json")
edges = jload("edges.json")
graph_stats = jload("graph_stats.json")

# v7c artifacts. The results.json holds feature importances, method comparisons,
# confusion matrices, bootstrap CIs, and conformal metrics. We also load the
# v6 feature CSVs for backward-compatible EDA on the test/cal splits.
v7c_results = json.loads((V7C / "results.json").read_text())

# Convenience aliases matching the v7c results structure.
sacred = {
    "version": "v7c",
    "best_method": "B_full_pipeline",
    "features": v7c_results["n_features"],
    "tier_accuracy": v7c_results["methods"]["B_full_pipeline"]["tier_accuracy"],
    "macro_f1": v7c_results["methods"]["B_full_pipeline"]["macro_f1"],
    "adjacent_accuracy": v7c_results["methods"]["B_full_pipeline"]["adjacent_accuracy"],
    "binary_accuracy": v7c_results["methods"]["B_full_pipeline"]["binary_accuracy"],
    "confusion_matrix": v7c_results["methods"]["B_full_pipeline"]["confusion_matrix"],
    "per_class": [
        {"tier": i, "count": v7c_results["methods"]["B_full_pipeline"]["class_counts"][t.lower()],
         "f1": v7c_results["methods"]["B_full_pipeline"]["per_class_f1"][t.lower()],
         "accuracy": (v7c_results["methods"]["B_full_pipeline"]["confusion_matrix"][t.lower()][t.lower()]
                      / v7c_results["methods"]["B_full_pipeline"]["class_counts"][t.lower()])}
        for i, t in enumerate(["Unrelated", "Partial", "Related", "Equivalent"])
    ],
    "n_test": sum(v7c_results["methods"]["B_full_pipeline"]["class_counts"].values()),
    "bootstrap_ci": {
        "acc_95": [v7c_results["bootstrap_ci_95"]["accuracy"]["lower"],
                   v7c_results["bootstrap_ci_95"]["accuracy"]["upper"]],
        "f1_95": [v7c_results["bootstrap_ci_95"]["macro_f1"]["lower"],
                  v7c_results["bootstrap_ci_95"]["macro_f1"]["upper"]],
    },
    "conformal": {
        "coverage": v7c_results["conformal"]["test_coverage"],
        "mean_set_size": v7c_results["conformal"]["avg_set_size"],
    },
}
sacred_path = V7C / "results.json"

# Load v6 test/cal features for EDA plots that still reference per-pair features.
# These contain the 22 v6 features; section 5 violin plots use a subset.
test_df = pd.read_csv(DATA / "v6_results" / "v6_test_features.csv")
cal_df = pd.read_csv(DATA / "v6_results" / "v6_cal_features.csv")
v6_results = json.loads((DATA / "v6_results" / "v6_all_results.json").read_text())

# Per-pair predictions (incl. conformal set for each test pair)
preds_df = pd.read_json(
    DATA / "v6_results" / "v6_pair_predictions.jsonl", lines=True
)

# Node2Vec UMAP projection (x, y per node, plus framework label)
n2v_proj = cload("node2vec_projection.csv")

# Build pandas views of the raw node/edge tables.
nodes_df = pd.DataFrame(nodes)
edges_df = pd.DataFrame(edges)

# Canonical framework order and pretty labels. Defined once in setup so every
# downstream section renders framework names consistently.
FRAMEWORKS = sorted(nodes_df["framework"].unique())
PRETTY = {
    "aiuc_1": "AIUC-1",
    "csa_aicm": "CSA AICM",
    "cosai_rm": "CoSAI RM",
    "eu_gpai_cop": "EU GPAI CoP",
    "mitre_atlas": "MITRE ATLAS",
    "nist_rmf": "NIST AI RMF",
    "owasp_agentic": "OWASP Agentic",
    "owasp_ai_exchange": "OWASP AI Exch.",
    "owasp_llm": "OWASP LLM",
}

# Explicit categorical palette for the nine frameworks. Graze & Schwabish
# (2024) recommend defining named palettes rather than relying on automatic
# color cycles, and limiting categorical palettes to distinguishable colors.
# Nine categories push the perceptual limit, so I use the Okabe-Ito
# colorblind-safe palette (Masataka Okabe & Kei Ito, 2002) which provides
# eight easily distinguishable colors, plus one additional muted purple.
FRAMEWORK_PALETTE = {
    "aiuc_1":           "#E69F00",   # orange
    "csa_aicm":         "#56B4E9",   # sky blue
    "cosai_rm":         "#009E73",   # bluish green
    "eu_gpai_cop":      "#F0E442",   # yellow
    "mitre_atlas":      "#0072B2",   # blue
    "nist_rmf":         "#D55E00",   # vermillion
    "owasp_agentic":    "#CC79A7",   # reddish purple
    "owasp_ai_exchange": "#999999",  # gray
    "owasp_llm":        "#882255",   # wine
}

print(f"nodes: {len(nodes_df):,}   edges: {len(edges_df):,}")
print(f"frameworks: {nodes_df['framework'].nunique()}")
print(f"orphan nodes (graph_stats): {len(graph_stats['orphan_nodes'])}")
print(f"v7c test pairs: {sacred['n_test']:,}   v7c features: {sacred['features']}")
print(f"sacred run: {sacred_path.name}  (version: {sacred.get('version','?')})")

## 3 · Dataset Overview: Schema, Marginals, Missingness

Before looking at any classifier output I want to establish what the underlying tables actually contain. This section answers six questions the COMP 4433 assignment asks of every exploratory analysis: how the continuous variables are distributed, how those variables relate to each other, what the central tendency looks like conditional on a categorical split, whether missing data is a concern, what the categorical composition of the corpus looks like, and which column would play the role of a target feature if I were building a predictive model of the crosswalk itself. Later sections reuse the tables introduced here, so this section is also the plain-data foundation the rest of the notebook builds on.

In [ ]:
# Schema profile and continuous-column summary. I first enrich both working
# DataFrames with the derived columns the rest of the analysis needs
# (description length in characters, parent-chain depth for nodes, a
# human-readable framework name, and the rationale-label length for edges),
# then render Figure 3.0 as a visual replacement for df.info() + df.describe()
# so the grader sees the schema graphically instead of as raw text. All
# derivations use pandas only; the figure uses matplotlib + seaborn, both on
# the COMP 4433 approved library list.

# Derived column 1: node description length. Missing descriptions are coded
# as NaN so the missing-data audit below can count them accurately. Edges
# do not carry a description column in this release; their verbosity is
# captured instead by the rationale_label character length.
nodes_df["desc_len"] = nodes_df["description"].fillna("").str.len()
nodes_df.loc[nodes_df["description"].isna(), "desc_len"] = np.nan
edges_df["rat_len"] = edges_df["rationale_label"].fillna("").str.len()
edges_df.loc[edges_df["rationale_label"].isna(), "rat_len"] = np.nan

# Derived column 2: parent-chain depth. A node's depth is the number of
# parent hops you can take before hitting the framework root. I compute it
# with an explicit Python loop because the recursion is at most 5 levels.
_parent_map = dict(zip(nodes_df["node_id"], nodes_df["parent_node_id"]))
def _depth(nid, seen=None):
    seen = set() if seen is None else seen
    d = 0
    while nid in _parent_map and pd.notna(_parent_map[nid]) and _parent_map[nid] in _parent_map:
        if nid in seen:
            break
        seen.add(nid)
        nid = _parent_map[nid]
        d += 1
        if d > 5:
            break
    return d
nodes_df["depth"] = [_depth(nid) for nid in nodes_df["node_id"]]

# Derived column 3: human-readable framework name (for plot labels).
nodes_df["framework_pretty"] = nodes_df["framework"].map(
    lambda f: f.replace("_", " ").title()
)

# ---- Figure 3.0 · Table profile -------------------------------------------
# Visual replacement for df.info() + df.describe() on both working tables.
# Top row: one horizontal bar per column, length = fill rate, color = dtype
# group, annotated with "dtype · n=<non-null count>" in the dtype color.
# Bottom row: horizontal five-number summary (IQR box, whiskers, white P50
# rule, coral mean diamond, numeric min/P50/mean/max annotations) for every
# continuous column in the two tables. Nothing here uses a library outside
# the COMP 4433 approved list.
DTYPE_COLORS = {
    "object": "#2a9d8f",   # teal   = strings
    "int":    "#6366f1",   # indigo = integers
    "float":  "#e9c46a",   # amber  = floats
    "bool":   "#e76f51",   # coral  = booleans
    "other":  "#9ca3af",   # gray   = datetimes / categoricals
}
def _dtype_group(dt):
    s = str(dt)
    if s == "object": return "object"
    if s.startswith("int"): return "int"
    if s.startswith("float"): return "float"
    if s == "bool": return "bool"
    return "other"

def _schema_bar(ax, df, title):
    cols = list(df.columns)
    fill_pct = np.array([df[c].notna().sum() / len(df) * 100 for c in cols])
    dtypes = [_dtype_group(df[c].dtype) for c in cols]
    order = np.argsort(-fill_pct)
    cols_o   = [cols[i] for i in order]
    dtypes_o = [dtypes[i] for i in order]
    fill_o   = fill_pct[order]
    y = np.arange(len(cols_o))
    colors = [DTYPE_COLORS[d] for d in dtypes_o]
    ax.barh(y, fill_o, color=colors, edgecolor="black", linewidth=0.4)
    ax.set_yticks(y)
    ax.set_yticklabels(cols_o, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlim(0, 155)
    ax.set_xlabel("% non-null")
    ax.set_title(
        f"{title}  ({df.shape[0]:,} rows × {df.shape[1]} cols)",
        fontsize=11, fontweight="bold", loc="left",
    )
    for i, (pct, dt, c) in enumerate(zip(fill_o, dtypes_o, cols_o)):
        n_nn = int(df[c].notna().sum())
        ax.text(
            min(pct + 2, 102), i,
            f"{dt} · n={n_nn:,}",
            va="center", fontsize=8,
            color=DTYPE_COLORS[dt], fontweight="bold",
        )
    ax.grid(axis="x", ls=":", lw=0.5, alpha=0.5)
    ax.set_axisbelow(True)

def _fivenum(ax, series, label, title, use_symlog=False, unit=""):
    s = series.dropna()
    q0, q25, q50, q75, q100 = np.percentile(s, [0, 25, 50, 75, 100])
    mean = float(s.mean())
    # Whisker from min to max.
    ax.plot([q0, q100], [0, 0], color="#9ca3af", lw=1.6, zorder=1)
    # IQR box.
    ax.add_patch(plt.Rectangle(
        (q25, -0.22), q75 - q25, 0.44,
        facecolor="#6366f1", edgecolor="black", lw=0.7, alpha=0.85, zorder=2,
    ))
    # Median rule.
    ax.plot([q50, q50], [-0.24, 0.24], color="white", lw=2.6, zorder=3)
    # Mean diamond.
    ax.plot([mean], [0], marker="D", color="#e76f51", markersize=9,
            markeredgecolor="black", markeredgewidth=0.6, zorder=4)
    # Numeric annotations above / below the axis of the row.
    ax.text(q0, -0.48, f"min\n{q0:,.0f}", fontsize=7,
            ha="left", va="top", color="#444")
    ax.text(q100, -0.48, f"max\n{q100:,.0f}", fontsize=7,
            ha="right", va="top", color="#444")
    ax.text(q50, 0.55,
            f"P50 = {q50:,.0f}    mean = {mean:,.1f}",
            fontsize=8, ha="center", va="bottom",
            fontweight="bold", color="#1c1c1c")
    ax.set_yticks([0])
    ax.set_yticklabels([label], fontsize=10)
    ax.set_ylim(-0.95, 0.95)
    if use_symlog:
        ax.set_xscale("symlog", linthresh=max(10, q50))
    span = max(q100 - q0, 1)
    ax.set_xlim(q0 - span * 0.08, q100 + span * 0.12)
    ax.set_xlabel(f"value ({unit})" if unit else "value")
    ax.set_title(title, fontsize=10, fontweight="bold", loc="left")
    ax.grid(axis="x", ls=":", lw=0.5, alpha=0.5)
    ax.set_axisbelow(True)

fig = plt.figure(figsize=(14, 8.5))
gs = gridspec.GridSpec(
    2, 6, figure=fig,
    height_ratios=[1.35, 1.0], hspace=0.75, wspace=1.2,
)

ax_a = fig.add_subplot(gs[0, 0:3])
_schema_bar(ax_a, nodes_df, "Figure 3.0A · nodes_df schema")

ax_b = fig.add_subplot(gs[0, 3:6])
_schema_bar(ax_b, edges_df, "Figure 3.0B · edges_df schema")

ax_c = fig.add_subplot(gs[1, 0:2])
_fivenum(
    ax_c, nodes_df["desc_len"], "desc_len",
    "Figure 3.0C · nodes_df.desc_len",
    use_symlog=True, unit="chars",
)

ax_d = fig.add_subplot(gs[1, 2:4])
_fivenum(
    ax_d, nodes_df["depth"], "depth",
    "Figure 3.0D · nodes_df.depth",
    use_symlog=False, unit="hops",
)

ax_e = fig.add_subplot(gs[1, 4:6])
_fivenum(
    ax_e, edges_df["rat_len"], "rat_len",
    "Figure 3.0E · edges_df.rat_len",
    use_symlog=True, unit="chars",
)

fig.suptitle(
    "Figure 3.0 · Table profile: schema (dtype + fill rate) and "
    "five-number summary per continuous column",
    fontsize=13, fontweight="bold", y=1.02,
)
plt.tight_layout()
plt.show()

Figure 3.0 is the at-a-glance profile of both tables: every column shows up as a horizontal bar (length encodes fill rate, color encodes dtype group), and the three bottom panels render the five-number summary for each continuous column without making the reader re-run `describe()`. The node table has 983 rows keyed by `node_id`, with framework membership, an `entry_type` tag (control, technique, risk, and so on), a free-text description, and an optional `parent_node_id` that encodes the intra-framework tree. The edge table has 5,813 rows with source and target node IDs, their framework affiliations, a confidence tag, a rationale code and human-readable rationale label, and several optional metadata columns populated only for the reviewed subset (`relevance`, `score`, `signals`, `notes`). The derived columns I added above give me three continuous variables to explore (`desc_len` and `depth` for nodes, `rat_len` for edges) and three primary categorical axes (`framework` and `entry_type` for nodes, `confidence` for edges). These are the variables the marginal distribution figure below is built on.

The horizontal bar layout encodes each column's non-null count as position along a common scale, the most accurate elementary perceptual task identified by Cleveland & McGill (1984). Color encodes a nominal variable (dtype), using distinct hues from a small categorical palette (Graze & Schwabish, 2024).

> **Plain English:** Think of this as the 'sticker on the side of the box' for the two spreadsheets the project runs on. Each bar shows one column: how long the bar is tells you how often that column is actually filled in, and the color tells you whether it holds words, whole numbers, decimals, or yes/no values. The three little summaries at the bottom show the shortest, middle, and longest values in the columns that hold numbers.

In [ ]:
# Figure 3.1. Four-panel marginal distribution figure answering COMP 4433
# guiding questions 1 (continuous distributions) and 3 (conditional central
# tendency). Uses a gridspec with differential column widths so the main
# distribution panel is the largest and the conditional box plot sits to
# the right as supporting evidence. Three plot types appear in one figure:
# histogram (top-left), KDE (top-right), horizontal bar with annotated
# medians (bottom-left), and box-per-framework (bottom-right). The
# on-plot annotation highlights the median node depth so the reader can
# read it without inspecting the axes.
fig = plt.figure(figsize=(14, 8))
gs = gridspec.GridSpec(
    2, 2, figure=fig,
    width_ratios=[1.35, 1.0], height_ratios=[1.0, 1.1],
    hspace=0.5, wspace=0.3,
)

# Panel A (top-left): histogram of node description length.
ax_a = fig.add_subplot(gs[0, 0])
len_vals = nodes_df["desc_len"].dropna()
ax_a.hist(
    len_vals, bins=40,
    color="#2a9d8f", edgecolor="white", linewidth=0.4,
)
ax_a.axvline(len_vals.mean(), color="#e76f51", ls="--", lw=1.2,
             label=f"mean = {len_vals.mean():.0f}")
ax_a.axvline(len_vals.median(), color="#264653", ls="-", lw=1.2,
             label=f"median = {len_vals.median():.0f}")
ax_a.set_xlabel("node description length (chars)")
ax_a.set_ylabel("count")
ax_a.set_title("Figure 3.1A · Node description length")
ax_a.legend(loc="upper right", fontsize=9)

# Panel B (top-right): KDE of node description length conditional on the
# five most common entry types. This answers guiding question 3: how does
# the central tendency shift when we condition on a categorical variable?
# Description length varies meaningfully across entry types because the
# underlying documents treat controls, techniques, and risks differently.
ax_b = fig.add_subplot(gs[0, 1])
et_nonnull = nodes_df.dropna(subset=["desc_len"])
top_entry_types = et_nonnull["entry_type"].value_counts().head(5).index.tolist()
palette_b = ["#264653", "#2a9d8f", "#e9c46a", "#e76f51", "#6366f1"]
for et, color in zip(top_entry_types, palette_b):
    sub = et_nonnull[et_nonnull["entry_type"] == et]["desc_len"]
    if len(sub) >= 5:
        sns.kdeplot(
            sub, ax=ax_b, color=color, lw=1.8, fill=True, alpha=0.18,
            label=f"{et} (n={len(sub)})",
            clip=(0, sub.quantile(0.98)),
        )
ax_b.set_xlabel("node description length (chars)")
ax_b.set_ylabel("density")
ax_b.set_title("Figure 3.1B · Node description length by entry type")
ax_b.legend(fontsize=8, loc="upper right")

# Panel C (bottom-left): horizontal bar of parent-chain depth distribution.
# Position on a common baseline is the most accurate perceptual channel for
# comparison, so a bar chart is the right tool for this discrete variable.
ax_c = fig.add_subplot(gs[1, 0])
depth_counts = nodes_df["depth"].value_counts().sort_index()
bars = ax_c.barh(
    [f"depth {d}" for d in depth_counts.index],
    depth_counts.values,
    color="#6366f1", edgecolor="black", linewidth=0.5,
)
for b, v in zip(bars, depth_counts.values):
    ax_c.text(v + 4, b.get_y() + b.get_height() / 2, str(int(v)),
              va="center", fontsize=9)
median_depth = nodes_df["depth"].median()
ax_c.annotate(
    f"median depth = {median_depth:.0f}",
    xy=(depth_counts.max() * 0.4, int(median_depth)),
    xytext=(depth_counts.max() * 0.55, int(median_depth) - 1.2),
    fontsize=10, fontweight="bold", color="#e76f51",
    arrowprops=dict(arrowstyle="->", color="#e76f51", lw=1.2),
)
ax_c.set_xlabel("node count")
ax_c.set_title("Figure 3.1C · Node depth in parent tree")

# Panel D (bottom-right): box plot of node description length per framework.
# This is the conditional central tendency view (guiding question 3) and
# also feeds back into the framework-landscape story in section 4.
ax_d = fig.add_subplot(gs[1, 1])
fw_order = (
    nodes_df.dropna(subset=["desc_len"])
    .groupby("framework")["desc_len"].median()
    .sort_values().index.tolist()
)
box_data = [
    nodes_df[nodes_df["framework"] == f]["desc_len"].dropna().values
    for f in fw_order
]
bp = ax_d.boxplot(
    box_data, vert=False, showfliers=False,
    patch_artist=True, widths=0.65,
)
for patch, color in zip(bp["boxes"], sns.color_palette("crest", n_colors=len(fw_order))):
    patch.set_facecolor(color)
    patch.set_edgecolor("black")
    patch.set_linewidth(0.5)
for med in bp["medians"]:
    med.set_color("black")
    med.set_linewidth(1.4)
ax_d.set_yticklabels([PRETTY.get(f, f) for f in fw_order], fontsize=9)
ax_d.set_xlabel("node description length (chars, fliers hidden)")
ax_d.set_title("Figure 3.1D · Description length by framework")

fig.suptitle(
    "Figure 3.1 · Marginal distributions and conditional central tendency",
    fontsize=14, fontweight="bold", y=1.01,
)
plt.tight_layout()
plt.show()

Node description length is right-skewed with a long tail: the median is a short phrase and a small minority of nodes carry multi-paragraph descriptions that pull the mean above the median. The parent-chain depth distribution is discrete and heavy at depth 1 and depth 2, which matches the typical catalogue shape of a two-level hierarchy where a top-level domain contains mid-level controls that contain individual activities. Only a handful of nodes reach depth 4 or 5, and those belong to NIST AI RMF, where the function-category-subcategory-activity tree is deepest. Conditioning description length on framework (panel D) shows that OWASP LLM and OWASP Agentic have the shortest descriptions because they are concise risk catalogues, while AIUC-1 and CSA AICM produce the longest descriptions because their entries are prose-style control statements with rationales attached. Panel B conditions node description length on `entry_type`: the five most common entry types (controls, techniques, risks, mitigations, and objectives) sit at visibly different central tendencies. Control-style entries are typically short and imperative, while risk- and technique-style entries tend to run longer because they carry explanatory prose. This conditional shift is one of the clearest visible associations between a categorical axis and a continuous feature in the dataset.

> **Plain English:** Most entries in the catalog are short, a sentence or two, and the hierarchy is mostly two levels deep, like a folder with subfolders. Some frameworks write in bullet points (OWASP), others in full paragraphs (AIUC-1, CSA AICM), and you can actually see that writing style show up as a difference in how long their entries are.

In [ ]:
# Figure 3.2. Missing-data audit. Two horizontal bar charts side by side
# (node columns on the left, edge columns on the right) showing the fraction
# of rows where each column is null. This is the most direct visualization
# for guiding question 4 (missing data).
node_nulls = nodes_df.isna().mean().sort_values(ascending=True)
edge_nulls = edges_df.isna().mean().sort_values(ascending=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

def _miss_bar(ax, series, title, color):
    bars = ax.barh(series.index, series.values * 100, color=color,
                   edgecolor="black", linewidth=0.4)
    for b, v in zip(bars, series.values):
        ax.text(v * 100 + 0.4, b.get_y() + b.get_height() / 2,
                f"{v*100:.1f}%", va="center", fontsize=8)
    ax.set_xlabel("% missing")
    ax.set_xlim(0, max(series.max() * 100 * 1.25, 10))
    ax.set_title(title)

_miss_bar(ax1, node_nulls, "Figure 3.2A · Missingness in nodes_df", "#2a9d8f")
_miss_bar(ax2, edge_nulls, "Figure 3.2B · Missingness in edges_df", "#6366f1")

# On-plot annotation: call out the biggest missing column.
worst_edge_col = edge_nulls.idxmax()
worst_pct = edge_nulls.max() * 100
if worst_pct > 1:
    ax2.annotate(
        f"largest gap:\n{worst_edge_col} ({worst_pct:.1f}%)",
        xy=(worst_pct, list(edge_nulls.index).index(worst_edge_col)),
        xytext=(worst_pct * 0.55, max(list(edge_nulls.index).index(worst_edge_col) - 1.5, 0)),
        fontsize=9, color="#e76f51", fontweight="bold",
        arrowprops=dict(arrowstyle="->", color="#e76f51", lw=1.0),
    )

fig.suptitle("Figure 3.2 · Missing-data audit", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

Missing data is concentrated in a narrow set of columns, and it is concentrated there for reasons that make sense. On the nodes table, `parent_node_id` is missing for every top-level entry, which is the correct semantics (a domain root has no parent). `description` is missing for a small number of placeholder entries that only carry a name. All the other node columns are complete. On the edges table, the optional metadata columns (`relevance`, `score`, `signals`, `notes`) are mostly empty because they are populated only for the reviewed subset of edges, and `rationale_label` is sparse for the same reason. The mandatory columns (`source_node_id`, `target_node_id`, `framework` slugs, `rationale_code`, `confidence`) are fully populated. The crucial point for the rest of this notebook is that **the v6 feature matrix has zero NaNs by construction**. The 22 features are all derived quantities (LLM scores, cosines, length statistics, and one-hot entry-type flags), and the feature-building pipeline imputes defaults for any upstream null before the classifier ever sees it. So the missing-data audit above is a fact about the raw tables, not a confound for the classifier. The one place where missingness still matters is downstream work: any follow-up analysis that wants to use rationale labels or reviewer metadata as a source of signal needs to restrict itself to reviewed edges, because those columns are empty on suggestive rows by design.

> **Plain English:** The blank cells in the raw tables are all in places where blanks are supposed to be there (like a top-level folder having no parent folder). The features the classifier actually uses are computed from those tables, not read directly, and that computation never hands the classifier a blank. So nothing in this missing-data picture is a problem for the model's scores later on.

In [ ]:
# Figure 3.3. Categorical composition of the node table. Three panels in a
# gridspec with differential row heights: a large stacked bar of entry_type
# by framework on top (guiding question 5), and two supporting bars on the
# bottom row showing the overall entry_type and confidence distributions.
fig = plt.figure(figsize=(13, 8))
gs = gridspec.GridSpec(
    2, 2, figure=fig,
    height_ratios=[1.6, 1.0], hspace=0.5, wspace=0.35,
)

# Panel A (top, spans both columns): stacked bar of entry_type per framework.
ax_a = fig.add_subplot(gs[0, :])
et_mat = (
    nodes_df.groupby(["framework", "entry_type"]).size().unstack(fill_value=0)
    .reindex(sorted(nodes_df["framework"].unique()))
)
et_mat = et_mat.loc[:, et_mat.sum().sort_values(ascending=False).index]
bottoms = np.zeros(len(et_mat))
palette = sns.color_palette("Set2", n_colors=len(et_mat.columns))
for i, col in enumerate(et_mat.columns):
    ax_a.bar(
        [PRETTY.get(f, f) for f in et_mat.index],
        et_mat[col].values, bottom=bottoms, label=col, color=palette[i],
        edgecolor="white", linewidth=0.4,
    )
    bottoms += et_mat[col].values
# Total-count labels above each stack. Cleveland & McGill (1984) note that
# only the bottom segment of a stacked bar shares a common baseline; the
# upper segments require the less-accurate length judgment. Adding numeric
# totals lets the reader compare framework sizes via the most accurate
# channel: reading exact values.
for j, total in enumerate(bottoms):
    ax_a.text(j, total + 2, str(int(total)),
              ha="center", va="bottom", fontsize=8, fontweight="bold")
ax_a.set_ylabel("node count")
ax_a.set_title("Figure 3.3A · Entry types per framework (absolute counts)")
plt.setp(ax_a.get_xticklabels(), rotation=25, ha="right")
ax_a.legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="entry type", fontsize=8)

# Panel B (bottom-left): overall entry_type distribution.
ax_b = fig.add_subplot(gs[1, 0])
et_total = nodes_df["entry_type"].value_counts()
sns.barplot(
    x=et_total.values, y=et_total.index,
    ax=ax_b, hue=et_total.index, palette="Set2", legend=False,
)
for i, v in enumerate(et_total.values):
    ax_b.text(v + 4, i, str(int(v)), va="center", fontsize=9)
ax_b.set_xlabel("node count")
ax_b.set_title("Figure 3.3B · Entry-type mix (overall)")

# Panel C (bottom-right): confidence distribution (edges).
ax_c = fig.add_subplot(gs[1, 1])
conf_total = edges_df["confidence"].fillna("unknown").value_counts()
sns.barplot(
    x=conf_total.values, y=conf_total.index,
    ax=ax_c, hue=conf_total.index, palette="crest", legend=False,
)
for i, v in enumerate(conf_total.values):
    ax_c.text(v + 80, i, str(int(v)), va="center", fontsize=9)
ax_c.set_xlabel("edge count")
ax_c.set_title("Figure 3.3C · Edge confidence mix")

fig.suptitle("Figure 3.3 · Categorical composition of the corpus",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

The stacked bar form in Figure 3.3A is a deliberate choice: the primary message is the total node count per framework, with entry-type composition as a secondary layer. Cleveland & McGill (1984) observe that only the bottom segment of a stacked bar shares a common baseline, making upper-segment comparisons less accurate. To mitigate this, numeric totals appear above each stack, giving the reader a position-based comparison channel alongside the area-based composition view.

Six guiding questions mapped to the figures above, for anyone who wants to tick them off explicitly.

1. **Distribution of continuous variables.** Figure 3.1A for node description length, Figure 3.1B for the same continuous variable stratified by entry type, and Figure 3.1C for node depth in the parent tree. All three distributions are right-skewed and the medians sit well below the means.

2. **Relationships between variables.** Figure 3.1B is a direct read of the relationship between a categorical variable (`entry_type`) and a continuous variable (`desc_len`): controls, techniques, risks, mitigations, and objectives each occupy a visibly different length regime. The correlation matrix of the v6-era legacy features in Figure 5.3 (later in the notebook) is the denser view of feature-to-feature relationships.

3. **Central tendency conditional on a categorical split.** Figure 3.1D shows node description length split by framework, with OWASP LLM at the bottom and CSA AICM at the top of the median ladder.

4. **Missing data.** Figure 3.2 quantifies it column by column. On the edge table the biggest gaps are in the optional metadata columns (`relevance`, `score`, `signals`, `notes`, `rationale_label`), all of which are populated only for the reviewed subset. The mandatory structural columns (framework slugs, node IDs, rationale_code, confidence) are complete. The feature matrix has zero NaNs, so nothing in the classifier analysis is confounded by these gaps.

5. **Categorical composition.** Figure 3.3 answers this in three pieces: the mix of entry types per framework (Figure 3.3A), the overall entry type distribution (Figure 3.3B), and the edge confidence distribution (Figure 3.3C).

6. **Target feature for a hypothetical predictive model.** The expert-labeled tier is the target for the classifier: a 4-class ordinal label (`Unrelated`, `Partial`, `Related`, `Equivalent`) attached to each candidate pair in the 150-pair calibration split and the 400-pair frozen test split. The rest of the notebook analyzes how well that classifier recovers this target, and Section 5 examines which of the 22 engineered features carry the most signal for it.

> **Plain English:** The raw data is two tables. One lists 983 security-control entries and the other lists 5,813 connections between them. Descriptions vary in length and a few entries have no description at all. Missing values are concentrated in edge descriptions for machine-proposed links, which is expected because those links have not been human-reviewed yet. The classifier works from 22 derived numbers per pair and none of those numbers are ever missing.

## 4 · The Dataset: Framework Landscape

The crosswalk is structurally lopsided in a way that affects every downstream analysis. AIUC-1 and CSA AICM together account for roughly half of all nodes, and AIUC-1 originates the overwhelming majority of cross-framework edges. Part of the explanation is that AIUC-1 was designed as a comprehensive control catalogue, so it naturally has many anchors that other frameworks can attach to. Part of the explanation is that the active labeling sessions concentrated their effort on AIUC-1 first because it offered the highest expected coverage per hour of SME review. Either way, any reader who treats the graph as if all frameworks contribute equally will be misled, and the figure below is designed to make the asymmetry impossible to miss in a single glance.

The heatmaps in this section combine **four data sources** to show the complete mapping landscape rather than just the pipeline's processed output:

1. **Graph edges** (`edges.json`) -- the processed crosswalk graph built by the mapping engine.
2. **Upstream mappings** (`mappings_v1.jsonl`) -- framework-to-framework mappings published upstream by OWASP that the graph-build pipeline has not yet ingested.
3. **Upstream cross-references** (`crossrefs_v1.jsonl`) -- cross-references between OWASP frameworks (e.g., Agentic to LLM Top 10).
4. **Pair-config anchors** (`config/pairs/*.yaml`) -- expert-validated or bootstrap-CV-pruned anchor pairs used by the classifier (e.g., CSA AICM to OWASP Agentic).

Any single source tells a partial story. The graph edges miss upstream OWASP mappings; the upstream files miss the pair-config anchors for CSA AICM to OWASP Agentic. Combining all four and deduplicating by (source, target) node pair gives the most honest picture of what has been mapped so far.

In [ ]:
# Canonical framework order and pretty labels. Sorting alphabetically by the
# internal slug keeps the heatmap reproducible across runs.
FRAMEWORKS = sorted(nodes_df["framework"].unique())
PRETTY = {
    "aiuc_1": "AIUC-1",
    "csa_aicm": "CSA AICM",
    "cosai_rm": "CoSAI RM",
    "eu_gpai_cop": "EU GPAI CoP",
    "mitre_atlas": "MITRE ATLAS",
    "nist_rmf": "NIST AI RMF",
    "owasp_agentic": "OWASP Agentic",
    "owasp_ai_exchange": "OWASP AI Exch.",
    "owasp_llm": "OWASP LLM",
}
labels = [PRETTY[f] for f in FRAMEWORKS]
fw_set = set(FRAMEWORKS)

# --- Build a unified cross-framework edge list from four sources ---
import yaml

# Source 1: processed graph edges (edges.json).
graph_cross = edges_df[edges_df["source_framework"] != edges_df["target_framework"]][
    ["source_framework", "target_framework", "source_node_id", "target_node_id"]
].copy()

# Source 2: upstream mappings (mappings_v1.jsonl).
UPSTREAM = REPO / "data" / "upstream"
upstream_map = pd.read_json(UPSTREAM / "mappings_v1.jsonl", lines=True)
upstream_map["source_node_id"] = upstream_map["source_framework"] + ":" + upstream_map["source_id"]
upstream_map = upstream_map[
    upstream_map["source_framework"].isin(fw_set)
    & upstream_map["target_framework"].isin(fw_set)
    & (upstream_map["source_framework"] != upstream_map["target_framework"])
][["source_framework", "target_framework", "source_node_id", "target_node_id"]]

# Source 3: upstream cross-references (crossrefs_v1.jsonl).
upstream_xref = pd.read_json(UPSTREAM / "crossrefs_v1.jsonl", lines=True)
upstream_xref["source_node_id"] = upstream_xref["source_framework"] + ":" + upstream_xref["source_id"]
upstream_xref = upstream_xref[
    upstream_xref["source_framework"].isin(fw_set)
    & upstream_xref["target_framework"].isin(fw_set)
    & (upstream_xref["source_framework"] != upstream_xref["target_framework"])
][["source_framework", "target_framework", "source_node_id", "target_node_id"]]

# Source 4: pair-config anchor pairs (mapping_engine/config/pairs/*.yaml).
# These are expert-validated or bootstrap-CV-pruned mappings that may not
# yet appear in edges.json (e.g. csa_aicm -> owasp_agentic).
anchor_rows = []
pairs_dir = REPO / "mapping_engine" / "config" / "pairs"
for yf in pairs_dir.glob("*.yaml"):
    with open(yf) as fh:
        cfg = yaml.safe_load(fh)
    if not cfg or "anchors" not in cfg:
        continue
    pairs_list = (cfg["anchors"] or {}).get("pairs", [])
    sf = cfg["source_framework"]
    tf = cfg["target_framework"]
    if sf not in fw_set or tf not in fw_set or sf == tf:
        continue
    for p in pairs_list:
        anchor_rows.append({
            "source_framework": sf,
            "target_framework": tf,
            "source_node_id": p["source"],
            "target_node_id": p["target"],
        })
anchor_df = pd.DataFrame(anchor_rows) if anchor_rows else pd.DataFrame(
    columns=["source_framework", "target_framework", "source_node_id", "target_node_id"]
)

cross_edges = (
    pd.concat([graph_cross, upstream_map, upstream_xref, anchor_df], ignore_index=True)
    .drop_duplicates(subset=["source_node_id", "target_node_id"])
)

print(f"Unified cross-framework edges: {len(cross_edges):,} (from {len(graph_cross):,} graph, {len(upstream_map):,} upstream, {len(upstream_xref):,} xrefs, {len(anchor_df):,} anchors)")

# Direction-agnostic edge matrix: each edge is counted for both
# participating frameworks so that risk catalogues show their connectivity.
directed_mat = (
    cross_edges.groupby(["source_framework", "target_framework"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=FRAMEWORKS, columns=FRAMEWORKS, fill_value=0)
)
edge_mat = (directed_mat + directed_mat.T).copy()

node_counts = (
    nodes_df.groupby("framework").size().reindex(FRAMEWORKS).sort_values(ascending=True)
)
node_counts.index = [PRETTY[f] for f in node_counts.index]

conf_counts = edges_df["confidence"].fillna("unknown").value_counts()
conf_order = ["authoritative", "expert", "suggestive", "unvalidated", "unknown"]
conf_counts = conf_counts.reindex([c for c in conf_order if c in conf_counts.index])

In [ ]:
# Figure 4.1. Composed three-panel layout. Gridspec rather than subplots
# because the heatmap carries the central message and deserves the largest
# share of the canvas, while the two bar charts are supporting evidence and
# can be smaller. This is the differentially-sized axes layout that the
# assignment asks for.
fig = plt.figure(figsize=(13, 9))
gs = gridspec.GridSpec(
    2, 2,
    width_ratios=[2.2, 1.0],
    height_ratios=[1.6, 1.0],
    hspace=0.45, wspace=0.35,
)

ax_h = fig.add_subplot(gs[0, :])
sns.heatmap(
    edge_mat.values,
    ax=ax_h,
    annot=True, fmt="d",
    cmap="crest",
    xticklabels=labels, yticklabels=labels,
    cbar_kws={"label": "cross-framework edges"},
)
ax_h.set_title("Figure 4.1 \u00b7 Cross-framework edge counts (direction-agnostic)")
ax_h.set_xlabel("partner framework")
ax_h.set_ylabel("framework")
plt.setp(ax_h.get_xticklabels(), rotation=30, ha="right")

# Annotation: AIUC-1 row. The eye should land here before it parses cells.
aiuc_row = FRAMEWORKS.index("aiuc_1")
ax_h.annotate(
    "AIUC-1 is the hub:\nmost edges connect here",
    xy=(len(FRAMEWORKS) - 0.5, aiuc_row + 0.5),
    xytext=(len(FRAMEWORKS) + 0.6, aiuc_row + 1.4),
    fontsize=9, ha="left", va="center",
    arrowprops=dict(arrowstyle="->", color="black", lw=1.0),
    annotation_clip=False,
)

# Bottom left: horizontal bar chart for nodes per framework.
ax_n = fig.add_subplot(gs[1, 0])
sns.barplot(
    x=node_counts.values, y=node_counts.index,
    ax=ax_n, hue=node_counts.index, palette="crest", legend=False,
)
ax_n.set_title("Nodes per framework")
ax_n.set_xlabel("node count")
ax_n.set_ylabel("")
for i, v in enumerate(node_counts.values):
    ax_n.text(v + 4, i, str(int(v)), va="center", fontsize=8)

# Bottom right: confidence histogram.
ax_c = fig.add_subplot(gs[1, 1])
sns.barplot(
    x=conf_counts.index, y=conf_counts.values,
    ax=ax_c, hue=conf_counts.index, palette="Set2", legend=False,
)
ax_c.set_title("Edges by confidence level")
ax_c.set_xlabel("")
ax_c.set_ylabel("edge count")
plt.setp(ax_c.get_xticklabels(), rotation=30, ha="right")
for i, v in enumerate(conf_counts.values):
    ax_c.text(i, v + 30, str(int(v)), ha="center", fontsize=8)

fig.suptitle("The crosswalk landscape: hub-and-spoke by design",
             y=1.00, fontsize=14, weight="bold")
plt.show()

The heatmap draws on four data sources: processed graph edges, upstream OWASP mappings, upstream cross-references, and pair-config anchor pairs validated through bootstrap CV pruning. Edge counts are direction-agnostic, so every framework shows its true connectivity. AIUC-1 remains the hub. CSA AICM now shows connections to OWASP Agentic (via 39 anchor pairs) in addition to its dense bidirectional link with AIUC-1. OWASP Agentic and OWASP LLM show connections to multiple frameworks. The confidence histogram on the right gives the appropriate skepticism prior: the majority of edges sit at the *suggestive* confidence level, meaning they were proposed by the mapping engine and have not been reviewed by an expert.

The heatmap panel uses the `crest` sequential colormap, a single-hue luminance ramp that is perceptually ordered (Borland & Taylor, 2007). This avoids the rainbow colormap pitfall where perceptual non-uniformity can introduce false boundaries in continuous data.

> **Plain English:** By combining graph edges, upstream OWASP mappings, cross-references, and pair-config anchors, every framework now shows its full connectivity. Previously invisible pairs like CSA AICM to OWASP Agentic now appear. The remaining zero cells are genuinely unmapped pairs where no data exists in any source.

### Transitive reachability: the mappings behind the zeros

Figure 4.1 counts only **direct edges** between framework pairs. Several cells read zero, notably MITRE ATLAS to CSA AICM. Yet these frameworks are clearly related: both address AI supply-chain risks, model evasion, and data poisoning. The zeros reflect a labeling gap, not a semantic gap. When two frameworks share no direct edge but both connect to a common bridge framework (typically AIUC-1), a **transitive (2-hop) path** exists between them.

Figure 4.1b below computes transitive reachability for every pair. For each node in framework A, it checks whether a path of length 1 (direct) or length 2 (through any bridge node) reaches any node in framework B. The count is the number of unique unordered (source, target) node pairs reachable by either route. This is the same metric shown in the interactive Dash app's "All reachability" heatmap toggle.

In [ ]:
# Figure 4.1b. Transitive reachability heatmap -- same layout as
# Figure 4.1's heatmap panel but showing unique node pairs reachable
# via direct edges OR 2-hop transitive paths through bridge frameworks.
# Resolve project2 data path (REPO may be project1/ or repo root)
_p2_derived = REPO / "project2" / "data" / "derived"
if not _p2_derived.exists():
    _p2_derived = REPO.parent / "project2" / "data" / "derived"
reach_path = _p2_derived / "pairwise_reachability.json"
with open(reach_path) as f:
    pairwise_reach = json.load(f)

# Build the reachability matrix (total = direct + transitive unique pairs)
reach_mat = pd.DataFrame(0, index=FRAMEWORKS, columns=FRAMEWORKS)
direct_mat = pd.DataFrame(0, index=FRAMEWORKS, columns=FRAMEWORKS)

for fw_a in FRAMEWORKS:
    for fw_b in FRAMEWORKS:
        if fw_a == fw_b:
            continue
        r = pairwise_reach.get(fw_a, {}).get(fw_b, {})
        reach_mat.loc[fw_a, fw_b] = r.get("total", 0)
        direct_mat.loc[fw_a, fw_b] = r.get("direct", 0)

n_unlocked = ((direct_mat == 0) & (reach_mat > 0)).sum().sum() // 2
n_disconnected = sum(
    1 for a in FRAMEWORKS for b in FRAMEWORKS
    if a < b and reach_mat.loc[a, b] == 0
)
print(f"Framework pairs with 0 direct but >0 transitive: {n_unlocked}")
print(f"Framework pairs with 0 connectivity in either mode: {n_disconnected}")

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    reach_mat.values.astype(int),
    ax=ax, annot=True, fmt="d", cmap="crest",
    xticklabels=labels, yticklabels=labels,
    cbar_kws={"label": "unique node pairs (direct + transitive)"},
)
ax.set_title("Figure 4.1b \u00b7 Cross-framework reachability (direct + transitive)")
ax.set_xlabel("partner framework")
ax.set_ylabel("framework")
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")

# Annotation: same position as Fig 4.1 AIUC-1 callout.
aiuc_row = FRAMEWORKS.index("aiuc_1")
ax.annotate(
    "Bridge paths fill\npreviously empty cells",
    xy=(len(FRAMEWORKS) - 0.5, aiuc_row + 0.5),
    xytext=(len(FRAMEWORKS) + 0.6, aiuc_row + 1.4),
    fontsize=9, ha="left", va="center",
    arrowprops=dict(arrowstyle="->", color="black", lw=1.0),
    annotation_clip=False,
)
plt.tight_layout()
plt.show()


Figure 4.1b uses the same layout and `crest` sequential colormap as the heatmap panel in Figure 4.1, but the counts now include transitive paths. If node A1 in MITRE ATLAS connects to node B1 in AIUC-1, and B1 connects to node C1 in CSA AICM, then (A1, C1) counts as a reachable pair even though no direct ATLAS-to-CSA edge exists. Comparing the two figures cell by cell, every value in 4.1b is greater than or equal to its counterpart in 4.1, and 18 previously empty cells now carry non-zero counts.

The effect is dramatic. CSA AICM, which had direct connections only to AIUC-1 and OWASP Agentic, now shows reachability to every other framework. The densest new connection is CSA AICM to OWASP AI Exchange (2,020 reachable pairs), mediated almost entirely through AIUC-1. MITRE ATLAS to CSA AICM, the pair that prompted this analysis, shows 629 reachable pairs despite zero direct edges.

The heatmap uses the `crest` sequential colormap, a single-hue luminance ramp that is perceptually ordered (Borland & Taylor, 2007). Cell-value annotations provide direct labeling to compensate for the low perceptual accuracy of color saturation alone (Cleveland & McGill, 1984, rank 6).

> **Plain English:** Many framework pairs show zero direct mappings, but that does not mean they are unrelated. By following two-hop paths through bridge frameworks (mostly AIUC-1), 18 of the 36 off-diagonal pairs gain connectivity they lacked before. Only three pairs remain completely disconnected: CoSAI to CSA AICM, CoSAI to OWASP AI Exchange, and CoSAI to EU GPAI. These are genuine coverage gaps where no mapping evidence exists in any source.

In [ ]:
# Figure 4.2. Grouped bar chart of entry-type composition by framework
# (row-normalized). Cleveland & McGill (1984) rank position along a common
# scale as the most accurate perceptual channel. The prior version used
# stacked bars where only the bottom segment shared a common baseline; upper
# segments required the less-accurate length-without-baseline judgment. Grouped
# bars place every entry-type category on the same baseline, making it easy
# to compare any single entry type across frameworks.
type_mat = (
    nodes_df.groupby(["framework", "entry_type"]).size().unstack(fill_value=0)
    .reindex(FRAMEWORKS)
)
type_mat = type_mat.div(type_mat.sum(axis=1), axis=0)
type_mat = type_mat.loc[:, type_mat.sum().sort_values(ascending=False).index]

n_groups = len(type_mat.index)
n_types = len(type_mat.columns)
x = np.arange(n_groups)
width = 0.8 / n_types
palette = sns.color_palette("Set2", n_colors=n_types)

fig, ax = plt.subplots(figsize=(13, 5.5))
for i, col in enumerate(type_mat.columns):
    offset = (i - n_types / 2 + 0.5) * width
    bars = ax.bar(
        x + offset, type_mat[col].values, width,
        label=col, color=palette[i], edgecolor="white", linewidth=0.4,
    )

ax.set_xticks(x)
ax.set_xticklabels([PRETTY[f] for f in type_mat.index], rotation=30, ha="right")
ax.set_title("Figure 4.2 \u00b7 Entry-type composition by framework (row-normalized, grouped)")
ax.set_ylabel("share of nodes")
ax.set_ylim(0, 1)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="entry type")
plt.tight_layout()
plt.show()

Figure 4.2 uses grouped bars rather than stacked bars so that every entry-type category sits on a common baseline. Cleveland & McGill (1984) rank position along a common scale as the most accurate elementary perceptual task; stacked bars sacrifice this for all but the bottom segment. The grouped layout makes it straightforward to compare, for example, the control share across all nine frameworks at a glance.

Frameworks differ from one another in the *kind* of entries they contain, not only in how many entries they have. AIUC-1 and CSA AICM are dominated by controls and the activity steps that implement those controls. MITRE ATLAS is mostly attack techniques and mitigations. NIST AI RMF decomposes into functions, categories, and subcategories. OWASP Agentic and OWASP LLM are short risk catalogues with around ten entries each. This composition asymmetry matters for the v7c classifier because several of its features are one-hot indicators for the source and target entry-type pair (`has_technique`, `has_mitigation`, `is_activity_subcategory`, `is_activity_risk`). A control-to-risk pair is a fundamentally different prediction problem than a technique-to-mitigation pair, and the classifier needs to know which one it is looking at before it can weight the other features appropriately.

> **Plain English:** Not all frameworks are the same size or shape. Some are encyclopedia-like control catalogues, others are short lists of risks. The model needs to know which type of pair it is looking at so it can judge similarity sensibly. Comparing a control to a risk is a very different question than comparing two controls.

## 5 · The v7c baseline and its failure

Before introducing data augmentation or a new architecture, I need to establish what the original v7c pipeline actually does — and where it breaks down.

v7c is a two-stage classifier. Stage one encodes each control as a node embedding using a graph attention network (GAT) trained on the crosswalk graph; stage two feeds 50 hand-engineered features into a logistic regression with L2 regularization (C=0.01). The 50 features come from three families: 35 GAT geometry features (cosine similarity, L2 distance, dot product, and 32 element-wise embedding diffs), 3 baseline text-and-graph features (BGE cosine, BM25, two-hop bridge score), and 12 cross-encoder soft probabilities from DeBERTa-v3-large, RoBERTa-large, and DeBERTa-v3-base.

On the frozen test set of 179 pairs the pipeline reaches **81.0% exact-match accuracy** and **0.512 macro F1**. Those numbers look reasonable until you look at the per-class breakdown: the Equivalent class achieves **F1 = 0.000**. The classifier never predicts Equivalent — not once across 179 test pairs. The 81% headline hides a total blind spot on the class that matters most for framework alignment.

In [ ]:
# Feature family definitions for v7c. The 50 features are split into three
# families: GAT embedding features, baseline text/graph features, and
# cross-encoder soft probabilities from three transformer models.
GAT_FEATS = (
    ["gat_cosine", "gat_l2", "gat_dot"]
    + [f"gat_diff_{i:02d}" for i in range(32)]
)
BASELINE_FEATS = ["bge_cosine", "bm25", "bridge"]
CE_FEATS = [
    "deberta_prob_0", "deberta_prob_1", "deberta_prob_2", "deberta_prob_3",
    "roberta_prob_0", "roberta_prob_1", "roberta_prob_2", "roberta_prob_3",
    "deberta_base_prob_0", "deberta_base_prob_1", "deberta_base_prob_2", "deberta_base_prob_3",
]
ALL_FEATS = GAT_FEATS + BASELINE_FEATS + CE_FEATS
FAMILY_COLOR = {"GAT": "#14b8a6", "Baseline": "#3b82f6", "CE": "#f59e0b"}

# CE model sub-groups for the feature importance plot.
CE_MODELS = {
    "DeBERTa-v3-large": ["deberta_prob_0", "deberta_prob_1", "deberta_prob_2", "deberta_prob_3"],
    "RoBERTa-large": ["roberta_prob_0", "roberta_prob_1", "roberta_prob_2", "roberta_prob_3"],
    "DeBERTa-v3-base": ["deberta_base_prob_0", "deberta_base_prob_1", "deberta_base_prob_2", "deberta_base_prob_3"],
}

# Legacy v6 feature lists (used for Section 5 violin plots on the v6 test CSV).
LLM_FEATS = [
    "llm_final_score", "llm_final_tier", "llm_confidence", "llm_is_unanimous",
    "llm_sonnet_1", "llm_sonnet_2", "llm_sonnet_3",
]
STRUCT_FEATS = [
    "depth_diff", "depth_src", "depth_tgt",
    "len_src", "len_tgt", "len_diff", "len_ratio",
    "n2v_cosine", "gat_cosine",
    "has_technique", "has_mitigation",
    "is_activity_subcategory", "is_activity_risk",
]
OPUS_FEATS = ["opus_score", "opus_confidence"]

TIER_ORDER = ["Unrelated", "Partial", "Related", "Equivalent"]

# Ordinal tier palette. Tiers are ordinal (Unrelated < Partial < Related <
# Equivalent), so Borner et al. (2019) and Borland & Taylor (2007) prescribe
# a luminance-varying sequential palette rather than categorically distinct
# hues. Lighter shades encode weaker relationships; the darkest shade marks
# the strongest (Equivalent). A single-hue blue-teal ramp is inherently
# colorblind-safe because it relies on luminance, not hue discrimination
# (Graze & Schwabish, 2024).
TIER_PALETTE = ["#c1d5e0", "#6ba3be", "#2b7a9e", "#0b3d5e"]

In [ ]:
# Figure 5.1. Small multiples of six representative features, two from each
# v6 family, broken out by expert tier. A violin plot is the right chart here
# because it shows the whole distributional shape, while the inner quartile
# lines anchor median and IQR without a separate box plot. These use the v6
# test features CSV; the v7c pipeline replaces LLM and Opus features with CE
# soft probabilities, but the structural features (gat_cosine, n2v_cosine)
# remain informative for EDA.
feat_panels = [
    ("llm_final_score", "LLM . final score (aggregated Sonnet vote)"),
    ("llm_confidence", "LLM . confidence"),
    ("n2v_cosine", "Structural . Node2Vec cosine"),
    ("gat_cosine", "Structural . GAT cosine"),
    ("opus_score", "Opus . score"),
    ("opus_confidence", "Opus . confidence"),
]

fig, axes = plt.subplots(2, 3, figsize=(13.5, 7.2), sharex=True)
for ax, (col, title) in zip(axes.flat, feat_panels):
    sns.violinplot(
        data=test_df, x="expert_tier", y=col,
        order=TIER_ORDER,
        hue="expert_tier", hue_order=TIER_ORDER,
        palette=TIER_PALETTE, inner="quartile", cut=0,
        linewidth=0.8, ax=ax, legend=False,
    )
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("")
    # Overlay per-tier means as black points so the reader can see the
    # central tendency at a glance, independent of the violin bandwidth.
    means = test_df.groupby("expert_tier")[col].mean().reindex(TIER_ORDER)
    ax.scatter(range(len(TIER_ORDER)), means.values,
               marker="o", color="black", s=28, zorder=5)

fig.suptitle("Figure 5.1 · v6 legacy feature distributions by expert tier",
             y=1.02, fontsize=14, weight="bold")
plt.tight_layout()
plt.savefig(REPO_ROOT / "report" / "figures" / "fig5_1_feature_violins.png", dpi=150, bbox_inches="tight")
plt.show()

Figure 5.1 uses the v6 legacy feature set (LLM votes, Opus scores, structural cosines) rather than the v7c cross-encoder features, because v7c replaced LLM and Opus with CE soft probabilities. Even on the legacy features the pattern is clear: `llm_final_score` separates Equivalent from the other tiers, but the Equivalent violin is narrow — only 7 test pairs. That small sample is the first sign of trouble. The GAT cosine panel shows partial overlap across all four tiers, which means geometry alone cannot resolve close cases.

The v7c pipeline reports logistic regression coefficients as feature importance (absolute value of the learned weights, normalized to sum to 1.0). This tells me which features the trained classifier actually uses to make decisions. The ablation study complements feature importance by comparing four methods: A (GAT-only), B (full 50-feature pipeline), C (CE-only), and D (raw CE average). Taken together, these two views tell the reader which individual features matter most and which families are redundant.

In [ ]:
# Figure 5.2. Feature importance + cumulative importance, side by side with
# differentially sized panels. The importance bar chart is the headline and
# gets the wider left panel; the cumulative curve is a supporting panel on
# the right. Features are colored by family so the three-source architecture
# is visually obvious.
fi = v7c_results["feature_importance"]
fi_series = pd.Series(fi).sort_values(ascending=True)

def feature_family_v7c(name):
    if name in CE_FEATS: return "CE"
    if name in BASELINE_FEATS: return "Baseline"
    return "GAT"

bar_colors = [FAMILY_COLOR[feature_family_v7c(n)] for n in fi_series.index]

fig = plt.figure(figsize=(14, 12))
gs = gridspec.GridSpec(1, 2, figure=fig, width_ratios=[2.0, 1.0], wspace=0.35)

ax_bar = fig.add_subplot(gs[0, 0])
ax_bar.barh(fi_series.index, fi_series.values,
            color=bar_colors, edgecolor="black", linewidth=0.35)
ax_bar.set_xlabel("Feature importance (normalized abs. LogReg weight)")
ax_bar.set_title("Figure 5.2A · Per-feature importance (v7c, 50 features)")
ax_bar.tick_params(axis="y", labelsize=7)

# Annotate the top feature with an arrow so the reader immediately sees the
# winner.
top_feat = fi_series.idxmax()
top_val = fi_series.max()
ax_bar.annotate(
    f"Top feature:\n{top_feat} ({top_val:.1%})",
    xy=(top_val, list(fi_series.index).index(top_feat)),
    xytext=(top_val * 0.55, list(fi_series.index).index(top_feat) - 4),
    fontsize=10, ha="left",
    arrowprops=dict(arrowstyle="->", lw=1.1, color="black"),
)
legend_elems = [Patch(facecolor=c, edgecolor="black", label=k)
                for k, c in FAMILY_COLOR.items()]
ax_bar.legend(handles=legend_elems, loc="lower right", frameon=True)

# Cumulative importance on the right. Sorted descending so the x-axis reads
# as 'how many features to reach X% of the total importance'.
ax_cum = fig.add_subplot(gs[0, 1])
fi_desc = fi_series.sort_values(ascending=False)
cum = np.cumsum(fi_desc.values) / fi_desc.values.sum()
ax_cum.plot(range(1, len(cum) + 1), cum, marker="o", color="#264653", lw=1.8,
            markersize=4)
ax_cum.axhline(0.80, color="#e76f51", ls="--", lw=1.0)
ax_cum.text(len(cum) * 0.55, 0.82, "80% of total importance",
            color="#e76f51", fontsize=9)
ax_cum.set_xlabel("top-k features")
ax_cum.set_ylabel("cumulative importance (fraction)")
ax_cum.set_ylim(0, 1.02)
ax_cum.set_title("Figure 5.2B · Cumulative")

plt.tight_layout()
plt.savefig(REPO_ROOT / "report" / "figures" / "fig5_2_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

The CE features dominate: the top-ranked feature is a DeBERTa-v3-large soft probability, and CE features collectively account for most of the cumulative importance. The 80% threshold (dashed orange line) is reached in roughly the top 10 features. That concentration tells me the 35 GAT geometry features are largely redundant — the cross-encoders have already compressed the semantic signal. This will matter when I redesign the architecture in Section 9.

In [ ]:
# Figure 5.3. Method comparison bar chart (v7c ablation). Each bar is a
# different method evaluated on the same frozen test set. Method B is the
# full 50-feature pipeline. Colors encode the feature family used.
methods = v7c_results["methods"]
method_data = []
for key in ["A_gat_only", "B_full_pipeline", "C_ce_only", "D_raw_ce_avg"]:
    m = methods[key]
    method_data.append({"name": m["label"], "accuracy": m["tier_accuracy"],
                        "macro_f1": m["macro_f1"], "key": key})
abl_df = pd.DataFrame(method_data)

def method_color(key):
    if key == "B_full_pipeline": return "#6366f1"
    if key == "A_gat_only": return FAMILY_COLOR["GAT"]
    if key == "C_ce_only": return FAMILY_COLOR["CE"]
    return "#9333ea"

fig, ax = plt.subplots(figsize=(10, 5.5))
bar_colors_abl = [method_color(k) for k in abl_df["key"]]
bars = ax.bar(abl_df["name"], abl_df["accuracy"],
              color=bar_colors_abl, edgecolor="black", linewidth=0.5)

# Reference lines: random baseline (25%) and majority baseline.
majority = max(methods["B_full_pipeline"]["class_counts"].values()) / sacred["n_test"]
ax.axhline(0.25, color="lightgray", ls=":", lw=1.0,
           label=f"Random (25%)")
ax.axhline(majority, color="gray", ls="--", lw=1.2,
           label=f"Majority class ({majority:.0%})")

# Inline value labels.
for b, v in zip(bars, abl_df["accuracy"]):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.008,
            f"{v:.1%}", ha="center", fontsize=10, fontweight="bold")

# Annotate the full-pipeline winner.
best_idx = abl_df[abl_df["key"] == "B_full_pipeline"].index[0]
ax.annotate(
    "Full 50-d pipeline wins:\nGAT + baseline + CE ensemble",
    xy=(best_idx, abl_df.loc[best_idx, "accuracy"]),
    xytext=(best_idx - 1.5, abl_df.loc[best_idx, "accuracy"] + 0.06),
    fontsize=10, ha="left",
    arrowprops=dict(arrowstyle="->", lw=1.1, color="black"),
)

ax.set_ylabel("4-tier accuracy on frozen test")
ax.set_ylim(0, 0.95)
ax.set_title("Figure 5.3 · Method comparison (v7c, n=179)")
plt.setp(ax.get_xticklabels(), rotation=15, ha="right")
ax.legend(loc="upper left", frameon=True)
plt.tight_layout()
plt.savefig(REPO_ROOT / "report" / "figures" / "fig5_3_method_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

Figure 5.3 shows that Method B (full 50-feature pipeline) outperforms the ablations, but the gap between B and C (CE-only) is small — about two percentage points. Method A (GAT-only) drops sharply. This confirms the importance ranking: CE features carry the load, GAT features add a small margin, and the three baseline features contribute noise as much as signal.

### Frozen test results

The numbers below come from the single locked evaluation run. I set aside the test split before any hyperparameter search and never touched it again until the final sacred evaluation. Everything that follows in the notebook is measured against this same frozen set.

In [ ]:
# Print the sacred summary so the reader can compare narrative claims against
# the raw numbers without scrolling back to section 2.
print(f"Sacred run:       {sacred_path.name}  (version: {sacred.get('version','?')})")
print(f"Best method:      {sacred['best_method']}")
print(f"Features:         {sacred['features']}")
print(f"Tier accuracy:    {sacred['tier_accuracy']:.1%}")
print(f"Macro F1:         {sacred['macro_f1']:.4f}")
print(f"Adjacent acc:     {sacred['adjacent_accuracy']:.1%}")
print(f"Binary acc:       {sacred['binary_accuracy']:.1%}")
ci_acc = sacred["bootstrap_ci"]["acc_95"]
ci_f1 = sacred["bootstrap_ci"]["f1_95"]
print(f"Bootstrap 95% CI (acc): [{ci_acc[0]:.1%}, {ci_acc[1]:.1%}]")
print(f"Bootstrap 95% CI (F1):  [{ci_f1[0]:.4f}, {ci_f1[1]:.4f}]")
print(f"Conformal coverage: {sacred['conformal']['coverage']:.1%}  (alpha=0.10)")
print(f"Conformal mean set size: {sacred['conformal']['mean_set_size']:.2f}")
print(f"CV macro F1:      {v7c_results['cv_macro_f1']:.4f}")
print(f"Best C:           {v7c_results['best_C']}")

In [ ]:
# Figure 5.4. Confusion matrix + per-class accuracy in a two-panel layout with
# differentially sized axes. The confusion matrix is the headline and gets
# the wider panel. An on-plot annotation highlights the largest off-diagonal
# error so the main failure mode is immediately visible.
cm = sacred["confusion_matrix"]
cm_array = np.array([[cm[t1.lower()][t2.lower()] for t2 in TIER_ORDER]
                     for t1 in TIER_ORDER])

fig = plt.figure(figsize=(13, 5.5))
gs = gridspec.GridSpec(1, 2, figure=fig, width_ratios=[1.35, 1.0], wspace=0.3)

ax1 = fig.add_subplot(gs[0, 0])
sns.heatmap(cm_array, annot=True, fmt="d", cmap="Blues",
            xticklabels=TIER_ORDER, yticklabels=TIER_ORDER, ax=ax1,
            cbar_kws={"label": "pair count"})
ax1.set_xlabel("Predicted")
ax1.set_ylabel("True (expert)")
ax1.set_title("Figure 5.4A · Confusion matrix (v7c, n=179)")

# Find the largest off-diagonal error and point at it.
max_off, max_ij = 0, (0, 0)
for i in range(4):
    for j in range(4):
        if i != j and cm_array[i, j] > max_off:
            max_off, max_ij = cm_array[i, j], (i, j)
ax1.annotate(
    f"Dominant error:\n{TIER_ORDER[max_ij[0]]}>{TIER_ORDER[max_ij[1]]}\n({max_off} pairs)",
    xy=(max_ij[1] + 0.5, max_ij[0] + 0.5),
    xytext=(max_ij[1] + 1.35, max(max_ij[0] - 0.6, 0.2)),
    fontsize=9.5,
    arrowprops=dict(arrowstyle="->", lw=1.0),
    color="white" if cm_array[max_ij] > cm_array.max() * 0.45 else "black",
)

ax2 = fig.add_subplot(gs[0, 1])
per_class_map = {p["tier"]: p for p in sacred["per_class"]}
accs = [per_class_map[i]["accuracy"] for i in range(4)]
counts = [per_class_map[i]["count"] for i in range(4)]
f1s = [per_class_map[i]["f1"] for i in range(4)]
bars = ax2.bar(TIER_ORDER, accs, color=TIER_PALETTE, edgecolor="black", linewidth=0.5)
ax2.set_ylabel("per-class accuracy")
ax2.set_title("Figure 5.4B · Per-class accuracy & F1")
ax2.set_ylim(0, 1.15)
for b, a, n, f in zip(bars, accs, counts, f1s):
    ax2.text(b.get_x() + b.get_width() / 2, a + 0.02,
             f"{a:.0%}\nF1={f:.2f}\nn={n}",
             ha="center", fontsize=8)

plt.tight_layout()
plt.savefig(REPO_ROOT / "report" / "figures" / "fig5_4_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

The Equivalent row in Figure 5.4A is all zeros. All 7 Equivalent test pairs are misclassified — 6 as Related, 1 as Partial. The per-class accuracy panel (5.4B) makes this explicit: Equivalent accuracy is 0%, F1 = 0.00. The dominant off-diagonal error visible in the heatmap annotation is the one the model gets most wrong, but the Equivalent failure is categorically different — it is not a boundary confusion, it is an invisible class.

In [ ]:
# Figure 5.5. Headline accuracy metrics against random and majority-class
# baselines. The bar labels print the exact percentage so the reader does not
# have to read off the y-axis.
metrics = {
    "4-Tier\naccuracy":   sacred["tier_accuracy"],
    "Adjacent\naccuracy": sacred["adjacent_accuracy"],
    "Binary\naccuracy":   sacred["binary_accuracy"],
}
majority_baseline = max(p["count"] for p in sacred["per_class"]) / sacred["n_test"]

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(metrics))
vals = list(metrics.values())
bar_colors = ["#264653", "#2a9d8f", "#e76f51"]
bars = ax.bar(x, vals, color=bar_colors, edgecolor="black", linewidth=0.5, width=0.55)

ax.axhline(majority_baseline, color="gray", ls="--", lw=1.2,
           label=f"Majority class ({majority_baseline:.0%})")
ax.axhline(0.25, color="lightgray", ls=":", lw=1.0,
           label="Random (25%)")

ax.set_xticks(x)
ax.set_xticklabels(list(metrics.keys()), fontsize=11)
ax.set_ylabel("accuracy")
ax.set_ylim(0, 1.05)
ax.set_title("Figure 5.5 · Accuracy metrics vs baselines (v7c)")
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.02, f"{v:.1%}",
            ha="center", fontsize=12, fontweight="bold")
ax.legend(loc="lower right", fontsize=10)
plt.tight_layout()
plt.savefig(REPO_ROOT / "report" / "figures" / "fig5_5_headline_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()

### The pivot

The 81% headline accuracy in Figure 5.5 looks competitive against the majority-class baseline, but it is built almost entirely on 172 non-Equivalent pairs. The 7 Equivalent pairs in the test set are invisible to the model — it has never seen enough of them to learn what they look like.

This is data starvation, not model weakness. The training set contains fewer than 30 Equivalent pairs across all splits. No amount of regularization tuning or feature engineering will fix a class with that few examples. I need more labeled Equivalent pairs.

This sent me looking for external sources of high-similarity control pairs. Where could I find more labeled data for the Equivalent class?

## 6 · Uncertainty quantification and ordinal structure

Two problems with the v7c baseline go beyond the Equivalent blind spot. First, the classifier produces a point prediction with no uncertainty estimate. A compliance workflow needs to know when the model is guessing. Second, the four tiers are ordered---an Unrelated-to-Equivalent error is worse than a Related-to-Equivalent error---but the logistic regression treats all misclassifications equally.

### Three directions worth investigating

Three changes would most likely improve the classifier:

1. **More Equivalent training data.** The expert training set contains only a handful of Equivalent pairs. Any external source of high-similarity control pairs could help.
2. **Ordinal regression instead of flat 4-class classification.** The logistic regression ignores the ordering of tiers. An ordinal loss that penalizes distant errors more heavily than adjacent ones could improve calibration.
3. **Conformal prediction for uncertainty.** Wrapping point predictions in prediction sets would give downstream consumers a calibrated measure of confidence.

The ordinal regression demonstrator below addresses direction 2. The OpenCRE discovery in the next section addresses direction 1. v_final combines both.

### Ordinal regression demonstrator

The four tiers (Unrelated < Partial < Related < Equivalent) form a natural ladder. An ordinal model estimates cumulative thresholds: P(tier >= k) for each cutpoint. This proof-of-concept uses `statsmodels.OrderedModel` on the calibration split to show what ordinal structure looks like in practice.

> **Plain English:** Instead of treating the four tiers as disconnected buckets, this model says "tier 0 is most likely, and as the feature score increases, the probability mass slides rightward toward tier 3." The S-curves below visualize that slide.

In [ ]:
# statsmodels OrderedModel on the calibration split. The model treats the
# four expert tiers as an ordered outcome, fits cut points between adjacent
# categories, and produces monotone cumulative probabilities. I restrict
# the fit to the five most important v6 features to keep the demonstrator
# focused and the convergence tight.
import warnings as _w
_w.filterwarnings("ignore", category=UserWarning, module="statsmodels")
from statsmodels.miscmodels.ordinal_model import OrderedModel

TOP5 = ["gat_cosine", "n2v_cosine", "opus_score", "llm_final_score", "llm_confidence"]
X_ord = cal_df[TOP5].values
y_ord = cal_df["expert_label"].astype(int).values

ord_mod = OrderedModel(y_ord, X_ord, distr="logit")
ord_res = ord_mod.fit(method="bfgs", disp=False, maxiter=400)

print("=== OrderedModel fit summary (top 5 v6 features) ===")
print(f"converged: {ord_res.mle_retvals['converged']}")
print(f"log-likelihood: {ord_res.llf:.2f}")
print(f"n_obs: {int(ord_res.nobs)}")
print()
print("Coefficients (positive pushes toward higher tier):")
for name, coef in zip(TOP5, ord_res.params[:len(TOP5)]):
    print(f"  {name:<20s}  {coef:+.3f}")
print()
print("Threshold parameters (between adjacent tiers):")
thresholds = ord_res.params[len(TOP5):]
for i, t in enumerate(thresholds):
    print(f"  threshold_{i}: {t:+.3f}")

In [ ]:
# Figure 6.1. Fitted cumulative probabilities for a grid of hypothetical
# pairs. I sweep a single feature (opus_score, the strongest signal) across
# its observed range while holding the other four features at their cal-split
# means, then plot the four cumulative tier probabilities as a stacked area
# chart. The chart visualizes how the ordinal model shifts mass from low
# tiers to high tiers as opus_score increases.
mean_vec = cal_df[TOP5].mean().values
opus_grid = np.linspace(
    cal_df["opus_score"].min(),
    cal_df["opus_score"].max(),
    60,
)
X_grid = np.tile(mean_vec, (len(opus_grid), 1))
opus_idx = TOP5.index("opus_score")
X_grid[:, opus_idx] = opus_grid
probs = ord_res.model.predict(ord_res.params, exog=X_grid)

# Figure 6.1 uses individual line curves rather than a stacked area chart.
# Cleveland & McGill (1984) rank position along a common scale (rank 1) above
# area (rank 4); line curves let the reader compare exact probability values
# at any opus_score by reading off the shared y-axis baseline, whereas stacked
# areas only allow accurate reading of the bottom layer.
fig, ax = plt.subplots(figsize=(10.5, 5.8))
for i, tier in enumerate(TIER_ORDER):
    ax.plot(
        opus_grid, probs[:, i],
        label=tier, color=TIER_PALETTE[i], lw=2.2, alpha=0.9,
    )
ax.set_xlim(opus_grid.min(), opus_grid.max())
ax.set_ylim(0, 1)
ax.set_xlabel("opus_score (other features at cal-split mean)")
ax.set_ylabel("predicted probability")
ax.set_title("Figure 6.1 · statsmodels OrderedModel: tier probabilities vs opus_score")
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), title="tier")

# On-plot annotation: mark where the modal prediction flips from Partial
# to Related, which is the key transition for this demonstrator.
mode_tier = np.argmax(probs, axis=1)
flip_idx = None
for i in range(1, len(mode_tier)):
    if mode_tier[i] != mode_tier[i - 1]:
        flip_idx = i
        break
if flip_idx is not None:
    ax.axvline(opus_grid[flip_idx], color="black", ls="--", lw=1.1)
    ax.annotate(
        f"modal-tier flip\n@ opus={opus_grid[flip_idx]:.2f}",
        xy=(opus_grid[flip_idx], 0.55),
        xytext=(opus_grid[flip_idx] + 0.3, 0.72),
        fontsize=9, fontweight="bold",
        arrowprops=dict(arrowstyle="->", lw=1.0, color="black"),
    )

plt.tight_layout()
plt.show()

The ordinal fit converges and the cumulative-probability plot shows a clean monotone sweep from mostly-Unrelated on the left to mostly-Equivalent on the right. This confirms that the ordinal structure in the data is real and exploitable. The v_final pipeline will use three ordinal loss functions that build on this principle.

> **Plain English:** The proof-of-concept works. When trained to respect the tier ordering, the model produces probability curves that slide smoothly from "definitely unrelated" to "probably equivalent" as the input feature increases. This motivated the ordinal losses used in the final pipeline.